<a href="https://colab.research.google.com/github/ddickson28/Captain-FPSO-BN/blob/Change-to-cumulative-damage/CumulativeDamage29_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mbnpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.3/105.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 107.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: plotly
    Found existing installation: plotly 5.24.1
    Uninstalling plotly-5.24.1:
      Successfully uninstalled plotly-5.24.1
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you

In [ ]:
#Import modules

from mbnpy import variable, cpm, inference #importing relevant code classes from MBNpy
import numpy as np

In [ ]:
"Building the BN and relationship"
from mbnpy import variable, cpm, inference #importing relevant code classes from MBNpy

#1 Define variables
# Wx- WeatherDamage Lx - LoadingDamage, Tx - TotalDamage, Px - PreviousRepair, TPrev - TotalDamagePrevious,
# Cx -CumulativeDamage , CLx - CrackLocationInterest,

varis = {}
causes = ['Wx','Lx','Tx']
interventions = ['Px']
effects = ['Cx']
observations = ['Clx']
resistance = ['Rx']
Auxilliary = ['Zx'] #Addition --> Completed
UniqueRV = ['Vx'] #Addition --> Completed
CommonRV = ['Ux'] #Addition --> Completed


n_components = 3

for i in range(n_components):
                varis[f'Wx{i+1}'] = variable.Variable(f'Wx{i+1}', ['0','0.2','0.4','0.6','0.8','1.0'] )

for i in range(n_components):
                varis[f'Lx{i+1}'] = variable.Variable(f'Lx{i+1}', ['0','0.2','0.4','0.6','0.8','1.0'] )

for i in range(n_components):
                varis[f'Tx{i+1}'] = variable.Variable(f'Tx{i+1}', ['0','0.2','0.4','0.6','0.8','1.0'] )

for i in range(n_components):
                varis[f'Px{i+1}'] = variable.Variable(f'Px{i+1}', ['False','True'] )

for i in range(-1, n_components):
                varis[f'Cx{i+1}'] = variable.Variable(f'Cx{i+1}', ['0','0.2','0.4','0.6','0.8','1.0'])

for i in range(n_components):
                varis[f'Clx{i+1}'] = variable.Variable(f'CLx{i+1}', ['False','True'])

for i in range(n_components):
                varis[f'Rx{i+1}'] = variable.Variable(f'Rx{i+1}', ['0.4','0.6','0.8','1.0','1.2','1.4'])

#Addition --> Completed
for i in range(n_components):
                varis[f'Zx{i+1}'] = variable.Variable(f'Zx{i+1}', ['-0.5','-0.25','0.0','0.25','0.5'])

#Addition --> Completed
for i in range(n_components):
                varis[f'Vx{i+1}'] = variable.Variable(f'Vx{i+1}', ['-0.5','-0.25','0.0','0.25','0.5'])

#Addition --> Completed
varis['Ux'] = variable.Variable('Ux', ['-0.5','-0.25','0.0','0.25','0.5'])

#2 Define the cpm of the variables from 1 in order specified above.

cpms = {}

for i in range(n_components):
                cpms[f'Wx{i+1}'] = cpm.Cpm(
                         variables = [varis[f'Wx{i+1}']], # Use varis dictionary
                            no_child = 1,
                            C=np.array([[0],[1],[2],[3],[4],[5]], dtype=int),
                            p=np.array([0.017,0.435,0.518,0.03,0.0,0.0])
                )

for i in range(n_components):
                cpms[f'Lx{i+1}'] = cpm.Cpm(
                         variables = [varis[f'Lx{i+1}']],
                            no_child = 1,
                            C=np.array([[0],[1],[2],[3],[4],[5]], dtype=int),
                            p=np.array([0.122,0.677,0.198,0.002,0.0,0])
                )

for i in range(n_components):
                cpms[f'Tx{i+1}'] = cpm.Cpm(
                         variables = [varis[f'Tx{i+1}'], varis[f'Wx{i+1}'], varis[f'Lx{i+1}']], # Use varis dictionary for Wx
                            no_child = 1,
                            C=np.array([
                                                [0,0,0],
                                                [1,1,0],
                                                [2,2,0],
                                                [3,3,0],
                                                [4,4,0],
                                                [5,5,0],
                                                [1,0,1],
                                                [2,1,1],
                                                [3,2,1],
                                                [4,3,1],
                                                [5,4,1],
                                                [5,5,1],
                                                [0,0,2],
                                                [3,1,2],
                                                [4,2,2],
                                                [5,3,2],
                                                [5,4,2],
                                                [5,5,2],
                                                [3,0,3],
                                                [4,1,3],
                                                [5,2,3],
                                                [5,3,3],
                                                [5,4,3],
                                                [5,5,3],
                                                [4,0,4],
                                                [5,1,4],
                                                [5,2,4],
                                                [5,3,4],
                                                [5,4,4],
                                                [5,5,4],
                                                [5,0,5],
                                                [2,1,5],
                                                [5,2,5],
                                                [5,3,5],
                                                [5,4,5],
                                                [5,5,5],

                             ], dtype=int),
                             p=np.array([1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,])

)

for i in range(n_components):
                cpms[f'Px{i+1}'] = cpm.Cpm(
                                            variables = [varis[f'Px{i+1}']],
                                            no_child=1,
                                            C=np.array([[0],[1]], dtype=int),
                                            p=np.array([1.0, 0.0])

)

                cpms['Cx0'] = cpm.Cpm(
                         variables = [varis['Cx0']], # Use varis dictionary
                            no_child = 1,
                            C=np.array([[0],[1],[2],[3],[4],[5]], dtype=int),
                            p=np.array([1.0,0.0,0.0,0.0,0.0,0.0])
                )

for i in range (n_components):
                            cpms[f'Cx{i+1}'] = cpm.Cpm(
                                    variables = [varis[f'Cx{i+1}'], varis[f'Px{i+1}'], varis[f'Cx{i}'], varis[f'Tx{i+1}']],
                                    no_child=1,
                                    C=np.array([[0,0,0,0],
                                                [0,1,0,0],
                                                [1,0,1,0],
                                                [1,1,1,0],
                                                [2,0,2,0],
                                                [0,1,2,0],
                                                [3,0,3,0],
                                                [0,1,3,0],
                                                [4,0,4,0],
                                                [0,1,4,0],
                                                [5,0,5,0],
                                                [0,1,5,0],
                                                [1,0,0,1],
                                                [1,1,0,1],
                                                [2,0,1,1],
                                                [1,1,1,1],
                                                [3,0,2,1],
                                                [1,1,2,1],
                                                [4,0,3,1],
                                                [1,1,3,1],
                                                [5,0,4,1],
                                                [1,1,4,1],
                                                [5,0,5,1],
                                                [1,1,5,1],
                                                [2,0,0,2],
                                                [2,1,0,2],
                                                [3,0,1,2],
                                                [2,1,1,2],
                                                [4,0,2,2],
                                                [2,1,2,2],
                                                [5,0,3,2],
                                                [2,1,3,2],
                                                [5,0,4,2],
                                                [2,1,4,2],
                                                [5,0,5,2],
                                                [2,1,5,2],
                                                [3,0,0,3],
                                                [3,1,0,3],
                                                [4,0,1,3],
                                                [3,1,1,3],
                                                [5,0,2,3],
                                                [3,1,2,3],
                                                [5,0,3,3],
                                                [3,1,3,3],
                                                [5,0,4,3],
                                                [3,1,4,3],
                                                [5,0,5,3],
                                                [3,1,5,3],
                                                [4,0,0,4],
                                                [4,1,0,4],
                                                [5,0,1,4],
                                                [4,1,1,4],
                                                [5,0,2,4],
                                                [4,1,2,4],
                                                [5,0,3,4],
                                                [4,1,3,4],
                                                [5,0,4,4],
                                                [4,1,4,4],
                                                [5,0,5,4],
                                                [4,1,5,4],
                                                [5,0,0,5],
                                                [5,1,0,5],
                                                [5,0,1,5],
                                                [5,1,1,5],
                                                [5,0,2,5],
                                                [5,1,2,5],
                                                [5,0,3,5],
                                                [5,1,3,5],
                                                [5,0,4,5],
                                                [5,1,4,5],
                                                [5,0,5,5],
                                                [5,1,5,5],
                        ], dtype=int),
                            p=np.array([1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])
)

#This needs to be updated to be conditioned on Z --> Completed structure, check R condition aligns ---> Completed
for i in range(n_components):
                cpms[f'Rx{i+1}'] = cpm.Cpm(
                         variables = [varis[f'Rx{i+1}'], varis[f'Zx{i+1}']],
                            no_child = 1,
                            C=np.array([[1,0],
                                        [1,1],
                                        [2,2],
                                        [3,3],
                                        [4,4],

                            ], dtype=int),
                            p=np.array([1.0,1.0,1.0,1.0,1.0])
)

for i in range (n_components):
                            cpms[f'Clx{i+1}'] = cpm.Cpm(
                                     variables = [varis[f'Clx{i+1}'], varis[f'Cx{i+1}'], varis[f'Rx{i+1}']],
                                     no_child=1,
                                     C=np.array([[0,0,0],
                                                 [0,1,0],
                                                 [1,2,0],
                                                 [1,3,0],
                                                 [1,4,0],
                                                 [1,5,0],
                                                 [0,0,1],
                                                 [0,1,1],
                                                 [0,2,1],
                                                 [1,3,1],
                                                 [1,4,1],
                                                 [1,5,1],
                                                 [0,0,2],
                                                 [0,1,2],
                                                 [0,2,2],
                                                 [0,3,2],
                                                 [1,4,2],
                                                 [1,5,2],
                                                 [0,0,3],
                                                 [0,1,3],
                                                 [0,2,3],
                                                 [0,3,3],
                                                 [0,4,3],
                                                 [1,5,3],
                                                 [0,0,4],
                                                 [0,1,4],
                                                 [0,2,4],
                                                 [0,3,4],
                                                 [0,4,4],
                                                 [0,5,4],
                                                 [0,0,5],
                                                 [0,1,5],
                                                 [0,2,5],
                                                 [0,3,5],
                                                 [0,4,5],
                                                 [0,5,5],
                                 ], dtype=int),
                                 p=np.array([1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0,   ])
                            )

#Addition conditioned on U & V --> Completed
for i in range(n_components):
                cpms[f'Zx{i+1}'] = cpm.Cpm(
                         variables = [varis[f'Zx{i+1}'],varis[f'Vx{i+1}'],varis['Ux']],
                            no_child = 1,
                            C=np.array([[0,0,0],
                                        [0,1,0],
                                        [1,2,0],
                                        [2,3,0],
                                        [2,4,0],
                                        [0,0,1],
                                        [1,1,1],
                                        [2,2,1],
                                        [2,3,1],
                                        [3,4,1],
                                        [1,0,2],
                                        [2,1,2],
                                        [2,2,2],
                                        [3,3,2],
                                        [4,4,2],
                                        [2,0,3],
                                        [2,1,3],
                                        [3,2,3],
                                        [4,3,3],
                                        [4,4,3],
                                        [3,0,4],
                                        [3,1,4],
                                        [4,2,4],
                                        [4,3,4],
                                        [4,4,4],
                            ], dtype=int), # Follow CPM from excel  --> Completed
                            p=np.array([1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,]) #Deterministic, set to 1 --> Completed
)

#Addition
for i in range(n_components):
                cpms[f'Vx{i+1}'] = cpm.Cpm(
                         variables = [varis[f'Vx{i+1}']],
                            no_child = 1,
                            C=np.array([[0],[1],[2],[3],[4],], dtype=int),
                            p=np.array([0.0,0.006,0.493,0.493,0.006,]) #Needs marginal probabilities
)

#Addition
cpms['Ux'] = cpm.Cpm(
                         variables = [varis['Ux']],
                            no_child = 1,
                            C=np.array([[0],[1],[2],[3],[4],], dtype=int),
                            p=np.array([0.0,0.006,0.493,0.493,0.006,]) #Needs marginal probabilities
)


#print(varis)
#print(cpms)

varis_elim_order = [varis[f"{c}{i}"] for c in causes for i in range(1, n_components + 1)  ] # Create a varis_elim_order list...
varis_elim_order += [varis[f"{k}{i}"] for k in interventions for i in range (1, n_components + 1)]
varis_elim_order += [varis['Ux']] # --> Completed
varis_elim_order += [varis[f"{v}{i}"] for v in UniqueRV for i in range (1, n_components +1)] # --> Completed
varis_elim_order += [varis[f"{z}{i}"] for z in Auxilliary for i in range (1, n_components +1)] # --> Completed
varis_elim_order += [varis[f"{r}{i}"] for r in resistance for i in range (1, n_components + 1)]
varis_elim_order += [varis[f"{o}{i}"] for o in observations for i in range (1, n_components)]
varis_elim_order += [varis[f"{e}{i}"] for e in effects for i in range (0, n_components +1)]
print(varis_elim_order)

marg_CrackLocationInterest = inference.variable_elim(cpms = cpms, var_elim = varis_elim_order, prod= True )

print(marg_CrackLocationInterest)



[<Variable representing Wx1['0', '0.2', '0.4', '0.6', '0.8', '1.0'] at 0x7f2927e85fd0>, <Variable representing Wx2['0', '0.2', '0.4', '0.6', '0.8', '1.0'] at 0x7f2947715970>, <Variable representing Wx3['0', '0.2', '0.4', '0.6', '0.8', '1.0'] at 0x7f2947715880>, <Variable representing Lx1['0', '0.2', '0.4', '0.6', '0.8', '1.0'] at 0x7f295c1a9040>, <Variable representing Lx2['0', '0.2', '0.4', '0.6', '0.8', '1.0'] at 0x7f292c797a10>, <Variable representing Lx3['0', '0.2', '0.4', '0.6', '0.8', '1.0'] at 0x7f2927c370b0>, <Variable representing Tx1['0', '0.2', '0.4', '0.6', '0.8', '1.0'] at 0x7f2927c36270>, <Variable representing Tx2['0', '0.2', '0.4', '0.6', '0.8', '1.0'] at 0x7f2927c35c70>, <Variable representing Tx3['0', '0.2', '0.4', '0.6', '0.8', '1.0'] at 0x7f29548ac6e0>, <Variable representing Px1['False', 'True'] at 0x7f2927c36ba0>, <Variable representing Px2['False', 'True'] at 0x7f292d66d0a0>, <Variable representing Px3['False', 'True'] at 0x7f295484b800>, <Variable representing U